# 07 — Hedefli Veri Toplama
gap_report.csv'deki bosluklari doldur. Her tur >= 2000 filme ulasana kadar targeted spider calistir.
Hucreyi durdurursan spider da durur. Tekrar baslatinca labels.csv'den kaldigi yerden devam eder.

In [2]:
from pathlib import Path

# --- Yerel ---
CODE_ROOT = Path('..').resolve()
DATA_ROOT = CODE_ROOT

# --- Colab ---
# from google.colab import drive
# drive.mount('/content/drive')
# CODE_ROOT = Path('/content/drive/MyDrive/film-genre-project')
# DATA_ROOT = Path('/content/drive/MyDrive/film-genre-project-data')

SCRAPY_DIR  = CODE_ROOT / 'src' / 'scraper'
SCRAPY_BIN  = CODE_ROOT / '.venv' / 'Scripts' / 'scrapy.exe'
IDS_FILE    = DATA_ROOT / 'tmdb_movie_ids.txt'
LABELS_CSV  = DATA_ROOT / 'labels.csv'
LABELS_V2   = DATA_ROOT / 'labels_v2.csv'
GAP_REPORT  = DATA_ROOT / 'gap_report.csv'
POSTERS_DIR = DATA_ROOT / 'posters'
LOG_PATH    = DATA_ROOT / 'targeted.log'

print('Scrapy bin :', SCRAPY_BIN.exists())
print('IDS file   :', IDS_FILE.exists())
print('Gap report :', GAP_REPORT.exists())
print('Labels     :', LABELS_CSV.exists())

Scrapy bin : True
IDS file   : True
Gap report : True
Labels     : True


## Mevcut Bosluklar

In [3]:
import pandas as pd

gap_df  = pd.read_csv(GAP_REPORT)
singles = gap_df[gap_df['type'] == 'single'].sort_values('deficit', ascending=False)

TARGETS = {}
print('Tur                  Mevcut  Hedef   Acik')
print('-' * 50)
total_deficit = 0
for _, row in singles.iterrows():
    TARGETS[row['combination']] = int(row['target_count'])
    mark = '<-- EKSIK' if row['deficit'] > 0 else ''
    print(f"  {row['combination']:20s} {int(row['current_count']):5,}  {int(row['target_count']):5,}  {int(row['deficit']):5,}  {mark}")
    total_deficit += row['deficit']
print('-' * 50)
print(f'Toplam acik: {int(total_deficit):,} film')

Tur                  Mevcut  Hedef   Acik
--------------------------------------------------
  History                532  2,000  1,468  <-- EKSIK
  Documentary            593  2,000  1,407  <-- EKSIK
  Animation              656  2,000  1,344  <-- EKSIK
  Mystery                830  2,000  1,170  <-- EKSIK
  Fantasy                960  2,000  1,040  <-- EKSIK
  Family               1,057  2,000    943  <-- EKSIK
  Science Fiction      1,146  2,000    854  <-- EKSIK
  Horror               1,401  2,000    599  <-- EKSIK
  Adventure            1,558  2,000    442  <-- EKSIK
  Crime                1,835  2,000    165  <-- EKSIK
  Action               2,227  2,000      0  
  Comedy               4,473  2,500      0  
  Drama                5,539  2,500      0  
  Romance              2,071  2,000      0  
  Thriller             2,529  2,000      0  
--------------------------------------------------
Toplam acik: 9,432 film


## Targeted Spider — Canli Izleme
Hucreyi durdurursan (Kernel > Interrupt) Scrapy da durur.  
Tekrar baslatmak icin bu hucreyi tekrar calistir — labels.csv'den kaldigi yerden devam eder.

In [3]:
import subprocess, time, datetime
from collections import Counter
from IPython.display import clear_output

POLL_INTERVAL = 15  # saniye

TARGET_GENRES = list(TARGETS.keys())


def _read_counts():
    try:
        df = pd.read_csv(LABELS_CSV, dtype={'tmdb_id': str})
        df['genres'] = df['genres'].apply(lambda x: x.split('|') if isinstance(x, str) else [])
        counts = Counter(g for gs in df['genres'] for g in gs if g in TARGETS)
        return len(df), counts
    except Exception:
        return 0, {}


proc = subprocess.Popen(
    [str(SCRAPY_BIN), 'crawl', 'targeted',
     '-a', f'ids_file={IDS_FILE}',
     '-s', f'LABELS_PATH={LABELS_CSV}',
     '-s', f'GAP_REPORT_PATH={GAP_REPORT}',
     '-s', f'IMAGES_STORE={POSTERS_DIR}',
     '-L', 'WARNING'],
    cwd=str(SCRAPY_DIR),
    stdout=open(LOG_PATH, 'a', encoding='utf-8'),
    stderr=subprocess.STDOUT,
)
print(f'targeted spider baslatildi — PID: {proc.pid}')

start_time          = time.time()
start_total, _      = _read_counts()
recent              = []

try:
    while proc.poll() is None:
        now              = time.time()
        total, counts    = _read_counts()
        elapsed          = now - start_time

        recent.append((now, total))
        recent = [(t, c) for t, c in recent if now - t <= 300]
        if len(recent) >= 2:
            dt = recent[-1][0] - recent[0][0]
            dc = recent[-1][1] - recent[0][1]
            speed_hr = dc / dt * 3600 if dt > 0 else 0
        else:
            speed_hr = 0

        done_genres = sum(1 for g in TARGETS if counts.get(g, 0) >= TARGETS[g])
        total_remaining = sum(max(0, TARGETS[g] - counts.get(g, 0)) for g in TARGETS)

        if speed_hr > 0:
            eta_sec  = total_remaining / speed_hr * 3600
            eta_str  = (datetime.datetime.now() + datetime.timedelta(seconds=eta_sec)).strftime('%d.%m %H:%M')
            eta_left = str(datetime.timedelta(seconds=int(eta_sec)))
        else:
            eta_str = eta_left = 'hesaplaniyor...'

        clear_output(wait=True)
        print('Targeted Spider — Canli Izleme')
        print('─' * 58)

        for g in sorted(TARGETS, key=lambda g: counts.get(g, 0) / TARGETS[g]):
            c   = counts.get(g, 0)
            t   = TARGETS[g]
            pct = min(c / t, 1.0)
            bar = '█' * int(20 * pct) + '░' * (20 - int(20 * pct))
            tag = 'OK' if c >= t else f'{t-c:,} eksik'
            print(f'  {g:20s} [{bar}] {pct*100:5.1f}%  ({c:,}/{t:,})  {tag}')

        print('─' * 58)
        print(f'  Tamamlanan tur      : {done_genres} / {len(TARGETS)}')
        print(f'  Toplam cekilmis     : {total:,}  (+{total - start_total:,} bu oturumda)')
        print(f'  Hiz                 : {speed_hr:,.0f} film/saat')
        print(f'  Kalan (tahmini)     : {total_remaining:,} film')
        print(f'  Kalan sure          : {eta_left}')
        print(f'  Tahmini bitis       : {eta_str}')
        print('─' * 58)
        print(f'  Gecen sure          : {str(datetime.timedelta(seconds=int(elapsed)))}')
        print(f'  PID: {proc.pid}  |  Log: {LOG_PATH.name}')
        print()
        print('  Hucreyi durdurursan Scrapy da durur.')
        print('  Tekrar baslatmak icin bu hucreyi tekrar calistir.')

        if done_genres == len(TARGETS):
            print(f'\n  Tum turler hedefe ulasti!')
            break

        time.sleep(POLL_INTERVAL)

except KeyboardInterrupt:
    proc.terminate()
    total, counts = _read_counts()
    print(f'\nDurduruldu — toplam {total:,} film cekildi.')

proc.wait()
total, _ = _read_counts()
print(f'\nTamamlandi. Toplam: {total:,} film')

Targeted Spider — Canli Izleme
──────────────────────────────────────────────────────────
  History              [████████░░░░░░░░░░░░]  42.9%  (858/2,000)  1,142 eksik
  Mystery              [███████████░░░░░░░░░]  58.4%  (1,168/2,000)  832 eksik
  Animation            [████████████░░░░░░░░]  63.3%  (1,266/2,000)  734 eksik
  Fantasy              [█████████████░░░░░░░]  66.7%  (1,334/2,000)  666 eksik
  Science Fiction      [██████████████░░░░░░]  73.3%  (1,466/2,000)  534 eksik
  Family               [████████████████░░░░]  84.9%  (1,698/2,000)  302 eksik
  Horror               [████████████████████] 100.0%  (2,004/2,000)  OK
  Documentary          [████████████████████] 100.0%  (2,017/2,000)  OK
  Adventure            [████████████████████] 100.0%  (2,017/2,000)  OK
  Crime                [████████████████████] 100.0%  (2,128/2,000)  OK
  Romance              [████████████████████] 100.0%  (2,269/2,000)  OK
  Action               [████████████████████] 100.0%  (2,615/2,000)  OK
  Th

## labels_v2.csv Olustur
Cekim bittikten sonra calistir.

In [4]:
import shutil

shutil.copy(LABELS_CSV, LABELS_V2)

with open(LABELS_V2, encoding='utf-8') as f:
    v2_count = sum(1 for _ in f) - 1

print(f'labels_v2.csv olusturuldu: {LABELS_V2}')
print(f'Toplam film: {v2_count:,}')

labels_v2.csv olusturuldu: C:\Users\Kerem\Desktop\MakinÖğrenmesi\film-genre-project\labels_v2.csv
Toplam film: 16,307
